###  Setup e Configuração
Verifica a existência da variáveis de ambiente, valida a conexão com ADLS e lista os arquivos dispiníveis no container raw/batch-data.

In [0]:
# INSTALAÇÃO DE BIBLIOTECAS E DEPENDÊNCIAS NECESSÁRIAS
%pip install python-dotenv azure-storage-file-datalake azure-identity pandas pyarrow pyodbc --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv, find_dotenv

dotenv_path = find_dotenv()

if dotenv_path:
    load_dotenv(dotenv_path, override=True)
    print(f"Variáveis de Ambiente encontradas no caminho:{dotenv_path} ")
else:
    print("Variáveis de Ambiente não encontradas.")

Variáveis de Ambiente encontradas no caminho:/Workspace/Users/vinicius.silveira144@gmail.com/estagio-empregadados-turma-2/.env 


In [0]:
adls_credentials = {
    "CLIENT_ID": os.getenv("CLIENT_ID"),
    "TENANT_ID": os.getenv("TENANT_ID"),
    "CLIENT_SECRET": os.getenv("CLIENT_SECRET"),
    "STORAGE_ACCOUNT_NAME": os.getenv("STORAGE_ACCOUNT_NAME"),
    "CONTAINER_NAME": os.getenv("CONTAINER_NAME", "raw")
}

jdbc_credentials = {
    "JDBC_HOST": os.getenv("JDBC_HOST"),
    "JDBC_DATABASE": os.getenv("JDBC_DATABASE"),
    "JDBC_USERNAME": os.getenv("JDBC_USERNAME"),
    "JDBC_PASSWORD": os.getenv("JDBC_PASSWORD")
}

print("\nStatus das Credenciais ADLS")

for k, v in adls_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")

print("\nStatus das Credenciais SQL Server")

for k, v in jdbc_credentials.items():
    print(f"  {k}: {'[DEFINIDO]' if v else '[NÃO DEFINIDO]'}")


all_defined  = all(
    list(adls_credentials.values()) +
    list(jdbc_credentials.values())
)

if all_defined :
    print("\nVariáveis de ambiente carregadas com sucesso!")
else:
    print("\nExistem variáveis de ambiente pendentes!")


Status das Credenciais ADLS
  CLIENT_ID: [DEFINIDO]
  TENANT_ID: [DEFINIDO]
  CLIENT_SECRET: [DEFINIDO]
  STORAGE_ACCOUNT_NAME: [DEFINIDO]
  CONTAINER_NAME: [DEFINIDO]

Status das Credenciais SQL Server
  JDBC_HOST: [DEFINIDO]
  JDBC_DATABASE: [DEFINIDO]
  JDBC_USERNAME: [DEFINIDO]
  JDBC_PASSWORD: [DEFINIDO]

Variáveis de ambiente carregadas com sucesso!


In [0]:
# Credenciais OAuth
adls_options = {
    f"fs.azure.account.auth.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["CLIENT_ID"],
    f"fs.azure.account.oauth2.client.secret.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": adls_credentials["CLIENT_SECRET"],
    f"fs.azure.account.oauth2.client.endpoint.{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net": f"https://login.microsoftonline.com/{adls_credentials["TENANT_ID"]}/oauth2/token"
}

batch_data_path = (
    f"abfss://{adls_credentials["CONTAINER_NAME"]}@{adls_credentials["STORAGE_ACCOUNT_NAME"]}.dfs.core.windows.net/"
    "batch-data"
)

files = (
    spark.read
    .format("binaryFile")
    .options(**adls_options)
    .load(batch_data_path)
    .select("path")
)

files.show(truncate=False)

+--------------------------------------------------------------------------------------------------+
|path                                                                                              |
+--------------------------------------------------------------------------------------------------+
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_itens_venda_caixa.csv     |
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/ecommerce_itens_pedido.csv         |
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/food_estoque_lojas.csv             |
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/physical_vendas_caixa.csv          |
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/food_avaliacoes_produto.csv        |
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/ecommerce_rastreamento_entregas.csv|
|abfss://raw@internshipdatalake.dfs.core.windows.net/batch-data/ecommerce_pedidos.csv      